# ESTR3114: Next Concept Prediction — joint versus PCGrad

Run the independent `another_experiment` on Google Colab. This notebook follows
the original notebook's GPU → clone → install → Drive → prepare → smoke test →
experiment → figures workflow. **Runtime → Change runtime type → GPU** first.

Push the new experiment to your GitHub repository before cloning. Run cells in
order. Main runs use validation only; the optional final-reporting cell is disabled
until you select settings. No output from the sentiment experiment is reused.

## 1. Clone the project and locate the independent experiment

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
import os

REPOSITORY_URL = "https://github.com/marcolo0714-lgtm/ESTR3114_Project"
CHECKOUT = Path("/content/estr3114-project")
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(CHECKOUT)], check=True)
elif not (CHECKOUT / ".git").is_dir():
    raise RuntimeError(f"Existing path is not a Git checkout: {CHECKOUT}")
else:
    print("Reusing checkout; files are not automatically overwritten or pulled.")

project = CHECKOUT / "another_experiment"
if not (project / "1_prepare_assets.py").is_file():
    raise RuntimeError("another_experiment is missing. Push the new files, then use an updated checkout.")
os.chdir(project)
print("Experiment:", project)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, check=True)


## 2. Install requirements and check the GPU

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first, then rerun this cell."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


## 3. Check scripts and run the small offline tests

In [ ]:
scripts = {
    "prepare": project / "1_prepare_assets.py",
    "train": project / "2_train.py",
    "experiment": project / "3_run_experiments.py",
    "analyze": project / "4_analyze.py",
    "predict": project / "5_predict.py",
}
for name, path in scripts.items():
    assert path.is_file(), path
    print(name, "->", path.name)

def run_script(name, *arguments):
    subprocess.run([sys.executable, "-u", str(scripts[name]), *map(str, arguments)], cwd=project, check=True)

subprocess.run([sys.executable, "-m", "pytest", "tests", "-q"], cwd=project, check=True)


## 4. Mount Google Drive and choose settings

Assets, smoke results, main results, and analysis have separate locations under
`MyDrive/ESTR3114_NCP`. Change `RUN_TAG` for an independent run or after changing
settings. Completed compatible runs are reused; incomplete runs are protected and
cannot resume midway. A changed asset configuration needs a fresh `ASSET_TAG`.

Start with 1,000 steps and batch size 2. No runtime estimate from the sentiment
experiment applies here. Checkpoints are optional and can consume many GB across
a sweep; enable `SAVE_MODELS` before training if you want prediction afterwards.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ESTR3114_NCP")
SEQUENCE_LENGTH = 256
MODEL_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
DATASET_REVISION = "main"  # resolved and recorded as an immutable commit during preparation
ASSET_TAG = f"gpt2_wikitext2_len{SEQUENCE_LENGTH}_v1"
RUN_TAG = "validation_v1"
STEPS = 1000             # change to 6000 for a longer independently trained budget
WEIGHTS = [0.3, 1.0, 3.0]
BETAS = [0.1]
SEEDS = [42, 43, 44]
BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
EVAL_EVERY = 100
DIAGNOSTICS_EVERY = 25
SAVE_MODELS = False

resources = DRIVE_ROOT / "assets" / ASSET_TAG
run_root = DRIVE_ROOT / "runs" / RUN_TAG
results = run_root / "results"
analysis = run_root / "analysis"
smoke_root = DRIVE_ROOT / "smoke" / RUN_TAG
run_root.mkdir(parents=True, exist_ok=True)
print("Assets:", resources)
print("Main results:", results)
print("Analysis:", analysis)
conditions = (2 + 2 * len(set(WEIGHTS)) * len(set(BETAS))) * len(set(SEEDS))
print("Main conditions:", conditions)


## 5. Prepare this experiment's own GPT-2 and dataset

The first run downloads assets. Later sessions reuse the manifest and local files.
No file from the sentiment experiment is referenced. A partial download/preparation
is left intact; choose a fresh asset tag if it did not complete.

In [ ]:
manifest_path = resources / "manifest.json"
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text())
    assert manifest["sequence_length"] == SEQUENCE_LENGTH, "Choose a new ASSET_TAG for changed sequence length."
    assert manifest["model_revision"] == MODEL_REVISION, "Model revision differs; choose a new ASSET_TAG."
    assert manifest["source"]["dataset"] == "Salesforce/wikitext"
    assert manifest["source"]["config"] == "wikitext-2-raw-v1"
    if DATASET_REVISION != "main":
        assert manifest["source"]["revision"] == DATASET_REVISION, "Dataset revision differs."
    print("Reusing prepared assets:", resources)
else:
    if resources.exists() and any(resources.iterdir()):
        raise RuntimeError("Partial assets found. Keep them and choose a fresh ASSET_TAG before retrying.")
    run_script("prepare", "--resources", resources,
               "--sequence-length", SEQUENCE_LENGTH,
               "--model-revision", MODEL_REVISION,
               "--dataset-revision", DATASET_REVISION)
    manifest = json.loads(manifest_path.read_text())

# Validate complete prepared bytes before the first training call.
import hashlib
def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert (resources / "tokenizer" / "tokenizer_config.json").is_file()
assert file_sha256(resources / "initial_model.pt") == manifest["initial_model_sha256"]
for split in ("train", "validation", "test"):
    assert file_sha256(resources / "data" / f"{split}.npz") == manifest["splits"][split]["sha256"]
print(json.dumps(manifest, indent=2))


## 6. Two-step GPU smoke test

This tests the actual prepared model with both optimizers. The smoke results stay
outside the main comparison. It does not evaluate the test split or save models.
If memory runs out, reduce batch sizes; PCGrad needs separate task gradients.

In [ ]:
for method in ("joint", "pcgrad"):
    run_script("train", "--method", method, "--weight", 1.0, "--beta", 0.1,
               "--seed", SEEDS[0], "--steps", 2, "--batch-size", 1,
               "--eval-batch-size", EVAL_BATCH_SIZE, "--eval-every", 2,
               "--diagnostics-every", 1, "--device", "cuda",
               "--resources", resources, "--output", smoke_root / method)


## 7. Run the validation experiment

This cell performs the configured full sweep and can take substantial time.
Keep the runtime connected. Results are written directly to Drive. Rerunning with
the same settings skips completed compatible conditions; it does not resume an
incomplete condition. Keep partial results and choose a new run tag if necessary.

In [ ]:
arguments = ["--weights", *WEIGHTS, "--betas", *BETAS, "--seeds", *SEEDS,
             "--steps", STEPS, "--batch-size", BATCH_SIZE,
             "--eval-batch-size", EVAL_BATCH_SIZE, "--eval-every", EVAL_EVERY,
             "--diagnostics-every", DIAGNOSTICS_EVERY, "--device", "cuda",
             "--resources", resources, "--output", results]
if SAVE_MODELS:
    arguments.append("--save-model")
run_script("experiment", *arguments)


## 8. Analyze results and display the figures (GPU not required)

In [ ]:
run_script("analyze", "--results", results, "--output", analysis, "--steps", STEPS)
from IPython.display import display, Image, Markdown
display(Markdown((analysis / "report.md").read_text()))
for figure in sorted(analysis.glob("*.png")):
    print(figure.name)
    display(Image(filename=str(figure)))


## 9. Optional final reporting after validation selection

First inspect validation results. Set the chosen lambda/beta below and enable
`RUN_FINAL_REPORT`. This trains fresh matched controls, joint, and PCGrad models
and evaluates their **final** checkpoints on the test split once per run. A separate
results folder keeps reporting apart from validation selection. The same selected
weights are used for both methods. This cell is skipped by default under Run all.

In [ ]:
RUN_FINAL_REPORT = False
SELECTED_WEIGHT = 1.0   # edit after validation; this is not a claimed optimum
SELECTED_BETA = 0.1
REPORT_TAG = "selected_v1"
report_root = DRIVE_ROOT / "reporting" / REPORT_TAG
if RUN_FINAL_REPORT:
    arguments = ["--weights", SELECTED_WEIGHT, "--betas", SELECTED_BETA,
                 "--seeds", *SEEDS, "--steps", STEPS, "--batch-size", BATCH_SIZE,
                 "--eval-batch-size", EVAL_BATCH_SIZE, "--eval-every", EVAL_EVERY,
                 "--diagnostics-every", DIAGNOSTICS_EVERY, "--device", "cuda",
                 "--resources", resources, "--output", report_root / "results", "--test"]
    if SAVE_MODELS:
        arguments.append("--save-model")
    run_script("experiment", *arguments)
    run_script("analyze", "--results", report_root / "results", "--output", report_root / "analysis")
    display(Markdown((report_root / "analysis" / "report.md").read_text()))
else:
    print("Final test reporting is disabled. Select weights using validation first.")


## 10. Optional checkpoint inspection

Enable `SAVE_MODELS` before training to create checkpoints. Choose any saved final
or best-validation file below. This cell skips gracefully when none was saved.

In [ ]:
CHECKPOINT = None  # optional: replace with Path("/content/drive/MyDrive/.../final.pt")
available = sorted(results.glob("*/checkpoints/final.pt"))
if RUN_FINAL_REPORT:
    available += sorted((report_root / "results").glob("*/checkpoints/final.pt"))
if CHECKPOINT is None and available:
    CHECKPOINT = available[0]
if CHECKPOINT is not None:
    print("Inspecting:", CHECKPOINT)
    run_script("predict", "--checkpoint", CHECKPOINT, "--resources", resources,
               "--text", "The scientist entered the laboratory", "--max-new-tokens", 20,
               "--device", "cuda")
else:
    print("No saved checkpoint. Enable SAVE_MODELS for a new training run if needed.")


## 11. Optional download of metrics and figures

All outputs already persist in Drive. Enable this cell to download a small ZIP
containing configurations, logs, summaries, and figures. Model checkpoints and
prepared assets are excluded. It includes the current main run and optional
reporting run, with separate archive folders.

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from zipfile import ZipFile, ZIP_DEFLATED
    from google.colab import files
    archive = Path("/content/ESTR3114_NCP_results.zip")
    sources = [("validation", run_root)]
    if RUN_FINAL_REPORT and report_root.exists():
        sources.append(("reporting", report_root))
    with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
        for label, base in sources:
            for path in base.rglob("*"):
                if path.is_file() and path.suffix in {".json", ".jsonl", ".csv", ".png", ".md"}:
                    bundle.write(path, str(Path(label) / path.relative_to(base)))
    files.download(str(archive))
else:
    print("Outputs remain in:", run_root)
